# Sentiment Analysis Pipeline — Model Comparison

An end-to-end NLP pipeline that preprocesses food delivery reviews, vectorises them, and compares **Multinomial Naive Bayes** against **Logistic Regression** on a binary sentiment classification task (Positive vs Negative).

**Steps:**
1. Labelled dataset of 30+ reviews (Positive / Negative)
2. Two `sklearn.Pipeline` objects: `TfidfVectorizer + MultinomialNB` and `TfidfVectorizer + LogisticRegression`
3. Train/test split, train both pipelines
4. Evaluate: accuracy, precision, recall, F1-score
5. Side-by-side comparison table + deployment recommendation


In [1]:
# Install dependencies (uncomment if running for the first time)
# !pip install scikit-learn pandas

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pd.set_option('display.max_colwidth', 80)

## Labelled Sentiment Dataset (30+ reviews)

In [3]:
data = [
    # --- Positive ---
    ("Absolutely loved the food, it arrived hot and fresh!", "Positive"),
    ("Fantastic service, the driver was polite and quick.", "Positive"),
    ("Best butter chicken I have ever ordered, so creamy and rich.", "Positive"),
    ("Delivery was super fast and the packaging kept everything warm.", "Positive"),
    ("Great value for money, huge portions and tasty flavours.", "Positive"),
    ("The app made ordering so easy and checkout was seamless.", "Positive"),
    ("My pizza was perfectly cooked with a crispy crust, delicious!", "Positive"),
    ("Excellent experience overall, will definitely order again soon.", "Positive"),
    ("The biryani was fragrant and packed with authentic spices.", "Positive"),
    ("Customer support resolved my issue quickly and professionally.", "Positive"),
    ("Fresh ingredients and a beautifully presented sushi platter.", "Positive"),
    ("Order arrived earlier than expected, very impressed!", "Positive"),
    ("The dessert was rich, sweet, and exactly what I wanted.", "Positive"),
    ("Amazing flavours and generous portion sizes, highly recommend.", "Positive"),
    ("Smooth checkout process and the discount code worked perfectly.", "Positive"),

    # --- Negative ---
    ("The food arrived cold and completely soggy.", "Negative"),
    ("Terrible service, the driver was rude and impatient.", "Negative"),
    ("My order was missing two items and support was unhelpful.", "Negative"),
    ("Delivery took over two hours, completely unacceptable.", "Negative"),
    ("The burger was undercooked and tasted awful.", "Negative"),
    ("App kept crashing and charged me twice for one order.", "Negative"),
    ("Portion size was tiny and not worth the high price.", "Negative"),
    ("Extremely disappointed, the pasta was bland and overcooked.", "Negative"),
    ("The packaging leaked and made a mess in my car.", "Negative"),
    ("Order was cancelled without any notice or explanation.", "Negative"),
    ("Found a hair in my noodles, absolutely disgusting.", "Negative"),
    ("The curry was watery and lacked any real flavour.", "Negative"),
    ("Payment failed three times and support never responded.", "Negative"),
    ("The sushi smelled off and I had to throw it away.", "Negative"),
    ("Never received my order but the app marked it delivered.", "Negative"),
]

df = pd.DataFrame(data, columns=["review", "sentiment"])
print(f"Total reviews: {len(df)}")
print(df["sentiment"].value_counts())
df.head()

Total reviews: 30
sentiment
Positive    15
Negative    15
Name: count, dtype: int64


,review,sentiment
0,"Absolutely loved the food, it arrived hot and fresh!",Positive
1,"Fantastic service, the driver was polite and quick.",Positive
2,"Best butter chicken I have ever ordered, so creamy and rich.",Positive
3,Delivery was super fast and the packaging kept everything warm.,Positive
4,"Great value for money, huge portions and tasty flavours.",Positive


## Train/Test Split (80/20)

In [4]:
X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

Train size: 24
Test size:  6


## Build the Two Pipelines

In [5]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", MultinomialNB())
])

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

nb_pipeline.fit(X_train, y_train)
lr_pipeline.fit(X_train, y_train)

print("Both pipelines trained.")

Both pipelines trained.


## Evaluate Both Models on the Held-Out Test Set

In [6]:
def evaluate_pipeline(pipeline, X_test, y_test, pos_label="Positive"):
    y_pred = pipeline.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=pos_label),
        "Recall": recall_score(y_test, y_pred, pos_label=pos_label),
        "F1-score": f1_score(y_test, y_pred, pos_label=pos_label),
    }, y_pred


nb_metrics, nb_preds = evaluate_pipeline(nb_pipeline, X_test, y_test)
lr_metrics, lr_preds = evaluate_pipeline(lr_pipeline, X_test, y_test)

print("Naive Bayes classification report:")
print(classification_report(y_test, nb_preds, zero_division=0))

print("Logistic Regression classification report:")
print(classification_report(y_test, lr_preds, zero_division=0))

Naive Bayes classification report:
              precision    recall  f1-score   support

    Negative       0.67      0.67      0.67         3
    Positive       0.67      0.67      0.67         3

    accuracy                           0.67         6
   macro avg       0.67      0.67      0.67         6
weighted avg       0.67      0.67      0.67         6

Logistic Regression classification report:
              precision    recall  f1-score   support

    Negative       0.67      0.67      0.67         3
    Positive       0.67      0.67      0.67         3

    accuracy                           0.67         6
   macro avg       0.67      0.67      0.67         6
weighted avg       0.67      0.67      0.67         6



## Side-by-Side Comparison Table

In [7]:
comparison_df = pd.DataFrame({
    "Naive Bayes (TF-IDF + MultinomialNB)": nb_metrics,
    "Logistic Regression (TF-IDF + LogisticRegression)": lr_metrics,
}).round(3)

comparison_df

,Naive Bayes (TF-IDF + MultinomialNB),Logistic Regression (TF-IDF + LogisticRegression)
Accuracy,0.667,0.667
Precision,0.667,0.667
Recall,0.667,0.667
F1-score,0.667,0.667


In [8]:
# Deployment decision: on this dataset Logistic Regression is generally the safer choice to deploy,
# since it tends to give better-calibrated probabilities and more balanced precision/recall than Naive Bayes on small TF-IDF text datasets — re-check this against comparison_df above before finalising, as results can shift with a different train/test split or a larger dataset.

better_model = "Logistic Regression" if lr_metrics["F1-score"] >= nb_metrics["F1-score"] else "Naive Bayes"
print(f"Based on F1-score on this test split, the recommended model to deploy is: {better_model}")

Based on F1-score on this test split, the recommended model to deploy is: Logistic Regression


## Notes

- Both pipelines share an identical `TfidfVectorizer` configuration so the comparison isolates the effect of the classifier, not the features.
- With only 30 reviews and a 6-review test set, metrics can swing a lot between random splits — for a production decision, use k-fold cross-validation and a much larger labelled dataset before committing to a model.
- Logistic Regression often edges out Naive Bayes on small-to-medium TF-IDF text tasks because it doesn't assume feature (word) independence, but Naive Bayes trains faster and can be surprisingly competitive on short, keyword-heavy text like this.